# 11 — DenseNet121 Training Recipe Screen (Fold 0)

Mục tiêu: screen nhanh một số thay đổi training recipe sau khi đã chốt architecture/resolution/preprocessing.

Giữ cố định:
- DenseNet121 ImageNet pretrained
- grayscale → RGB 3-channel
- 224×224
- batch 8 + gradient accumulation 2
- weighted CE baseline
- locked fold 0
- holdout không dùng

Chỉ thử 4 cấu hình:
1. **control** — recipe hiện tại
2. **no_colorjitter** — bỏ brightness/contrast jitter
3. **label_smoothing_005** — weighted CE + label smoothing 0.05
4. **lr_5e5** — LR 5e-5 thay vì 1e-4

Screen rule:
- so P@0.50 với **control chạy cùng notebook**
- chỉ promote candidate nếu tăng khoảng **>= +0.001**
- threshold-tuned P chỉ là tín hiệu phụ


In [ ]:
print('>>> CELL 1 START: config', flush=True)
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

from pathlib import Path
import json,time,random,gc
import numpy as np,pandas as pd
from PIL import Image,ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from sklearn.metrics import confusion_matrix,f1_score,roc_auc_score

SEED=2026
ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'screen_outputs'/'densenet_training_recipe'
OUT.mkdir(parents=True,exist_ok=True)

FOLD=0
IMAGE_SIZE=224
BATCH_SIZE=8
ACCUM=2
EPOCHS=8
PATIENCE=3
WD=1e-4
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'

# Make candidate comparisons more reproducible than previous exploratory runs.
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True

CANDIDATES=[
    {'name':'control','lr':1e-4,'color_jitter':True,'label_smoothing':0.0},
    {'name':'no_colorjitter','lr':1e-4,'color_jitter':False,'label_smoothing':0.0},
    {'name':'label_smoothing_005','lr':1e-4,'color_jitter':True,'label_smoothing':0.05},
    {'name':'lr_5e5','lr':5e-5,'color_jitter':True,'label_smoothing':0.0},
]

print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Batch:',BATCH_SIZE,'| accum:',ACCUM,'| effective:',BATCH_SIZE*ACCUM,flush=True)
print('Candidates:',[c['name'] for c in CANDIDATES],flush=True)


In [ ]:
print('>>> CELL 2 START: locked split + grayscale cache', flush=True)
manifest=pd.read_csv(MANIFEST)
dev=manifest[manifest.split=='dev'].copy().reset_index(drop=True)
holdout=manifest[manifest.split=='holdout'].copy().reset_index(drop=True)
assert len(dev)==4362 and len(holdout)==770
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
dev['cache_idx']=np.arange(len(dev))

tr=dev[dev.cv_fold!=FOLD].copy().reset_index(drop=True)
va=dev[dev.cv_fold==FOLD].copy().reset_index(drop=True)
assert set(tr.sha256).isdisjoint(set(va.sha256))

class PadToSquare:
    def __call__(self,img):
        w,h=img.size;s=max(w,h)
        l=(s-w)//2;r=s-w-l;t=(s-h)//2;b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pad=PadToSquare()
buf=[];t0=time.time()
for i,row in dev.iterrows():
    with Image.open(row.path) as im:
        im=im.convert('L').convert('RGB')
        im=pad(im).resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
        buf.append(np.asarray(im,dtype=np.uint8))
    if (i+1)%500==0 or i+1==len(dev):
        print(f'  cached {i+1}/{len(dev)}',flush=True)
DEV_CACHE=np.stack(buf)
del buf
print(f'Cache: {DEV_CACHE.shape} | {DEV_CACHE.nbytes/1024**3:.2f} GiB | {time.time()-t0:.0f}s',flush=True)
print('Train:',len(tr),'Val:',len(va),'SEALED holdout:',len(holdout),flush=True)


In [ ]:
print('>>> CELL 3 START: helpers', flush=True)
MEAN=[0.485,0.456,0.406];STD=[0.229,0.224,0.225]
L2I={'Normal':0,'Pneumonia':1}

def make_train_tf(use_color_jitter):
    ops=[
        transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    ]
    if use_color_jitter:
        ops.append(transforms.ColorJitter(brightness=0.08,contrast=0.08))
    ops += [transforms.ToTensor(),transforms.Normalize(MEAN,STD)]
    return transforms.Compose(ops)

val_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class CachedDS(Dataset):
    def __init__(self,df,tf): self.df=df.reset_index(drop=True);self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return self.tf(Image.fromarray(DEV_CACHE[int(r.cache_idx)])),L2I[r.label],r.filename

def score(y,p,t=.5,auc=None):
    y=np.asarray(y,int);p=np.asarray(p,float);pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y);se=tp/max(tp+fn,1);sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    if auc is None: auc=roc_auc_score(y,p)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),
            'F1':float(f1),'AUROC':float(auc),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'threshold':float(t)}

@torch.no_grad()
def predict(model,loader):
    model.eval();ys=[];ps=[];names=[]
    for x,y,n in loader:
        x=x.to(DEVICE)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            z=model(x)
        ps.extend(torch.softmax(z.float(),1)[:,1].cpu().numpy())
        ys.extend(y.numpy());names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def make_model():
    m=models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)


In [ ]:
print('>>> CELL 4 START: recipe screen', flush=True)
results=[]

for cfg in CANDIDATES:
    name=cfg['name']
    print('\n'+'='*80,flush=True)
    print('CANDIDATE:',name,flush=True)
    print(cfg,flush=True)
    print('='*80,flush=True)

    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    if USE_AMP: torch.cuda.manual_seed_all(SEED)

    train_tf=make_train_tf(cfg['color_jitter'])
    tl=DataLoader(CachedDS(tr,train_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=False)
    vl=DataLoader(CachedDS(va,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False)

    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

    model=make_model()
    c=tr.label.value_counts();n0=int(c.Normal);n1=int(c.Pneumonia);tot=n0+n1
    cw=torch.tensor([tot/(2*n0),tot/(2*n1)],dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=cw,label_smoothing=cfg['label_smoothing'])
    opt=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=WD)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best=-1;best_epoch=0;best_state=None;stale=0;hist=[]

    for epoch in range(1,EPOCHS+1):
        et=time.time();model.train();opt.zero_grad(set_to_none=True);ls=0.;seen=0
        for bi,(x,y,_) in enumerate(tl,1):
            x=x.to(DEVICE);y=y.to(DEVICE)
            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                z=model(x);raw=criterion(z,y);loss=raw/ACCUM
            scaler.scale(loss).backward()
            if bi%ACCUM==0 or bi==len(tl):
                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
            ls+=float(raw.item())*len(x);seen+=len(x)

            if bi%100==0 or bi==len(tl):
                print(f'  {name} epoch {epoch:02d} batch {bi}/{len(tl)} | loss={ls/seen:.4f}',flush=True)

        sched.step()
        yv,pv,_=predict(model,vl);m=score(yv,pv,.5);sec=time.time()-et
        hist.append({'candidate':name,'epoch':epoch,'train_loss':ls/seen,'seconds':sec,**m})
        print(f"{name} Epoch {epoch:02d} | P={m['P']:.5f} | A={m['Accuracy']:.5f} | "
              f"Se={m['Sensitivity']:.5f} | Sp={m['Specificity']:.5f} | "
              f"F1={m['F1']:.5f} | AUROC={m['AUROC']:.6f} | {sec:.0f}s",flush=True)

        if m['P']>best+1e-6:
            best=m['P'];best_epoch=epoch;stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE:
                print(name,'early stopping',flush=True);break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,vl)
    fixed=score(yv,pv,.5)
    auc=float(roc_auc_score(yv,pv))
    ths=np.round(np.arange(.10,.901,.005),3)
    scan=pd.DataFrame([score(yv,pv,float(t),auc) for t in ths])
    br=scan.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
    bt=float(br.threshold);tuned=score(yv,pv,bt,auc)

    pd.DataFrame(hist).to_csv(OUT/f'{name}_history.csv',index=False)
    pd.DataFrame({'filename':names,'y_true':yv,'p_pneumonia':pv}).to_csv(OUT/f'{name}_predictions.csv',index=False)
    scan.to_csv(OUT/f'{name}_threshold_scan.csv',index=False)

    results.append({
        'candidate':name,'best_epoch':best_epoch,'lr':cfg['lr'],
        'color_jitter':cfg['color_jitter'],'label_smoothing':cfg['label_smoothing'],
        'P_at_0_5':fixed['P'],'AUROC':fixed['AUROC'],
        'Sensitivity':fixed['Sensitivity'],'Specificity':fixed['Specificity'],'F1':fixed['F1'],
        'best_threshold':bt,'P_tuned_fold0':tuned['P']
    })

    del model,opt,sched,scaler,best_state,tl,vl
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

print('>>> SCREEN FINISHED',flush=True)


In [ ]:
print('>>> CELL 5 START: comparison', flush=True)
table=pd.DataFrame(results)
control=float(table.loc[table.candidate=='control','P_at_0_5'].iloc[0])
control_tuned=float(table.loc[table.candidate=='control','P_tuned_fold0'].iloc[0])

table['delta_vs_control_P_at_0_5']=table.P_at_0_5-control
table['delta_vs_control_tuned']=table.P_tuned_fold0-control_tuned
table=table.sort_values('P_at_0_5',ascending=False)
table.to_csv(OUT/'training_recipe_screen_summary.csv',index=False)
display(table)

eligible=table[(table.candidate!='control') & (table.delta_vs_control_P_at_0_5>=0.001)]
if len(eligible):
    winner=eligible.iloc[0]
    print('PROMOTE:',winner.candidate,'| delta=',winner.delta_vs_control_P_at_0_5,flush=True)
else:
    print('NO candidate gained >= +0.001 over same-run control. Keep current recipe.',flush=True)

with open(OUT/'screen_decision.json','w',encoding='utf-8') as f:
    json.dump({
        'control_P_at_0_5':control,
        'control_tuned_P':control_tuned,
        'promotion_threshold_delta':0.001,
        'eligible_candidates':eligible.to_dict(orient='records'),
        'holdout_used':False
    },f,indent=2)


## Sau notebook này

- Nếu có candidate tăng **>= +0.001 P@0.50** so với control cùng run → candidate đó mới đáng 5-fold.
- Nếu không có → dừng tuning recipe, chuyển sang multi-seed/final ensemble.
- Không mở holdout.
